# Pipeline 


                Feature Extraction
                        ↓
                Scaling
                        ↓
                Train/Test Split
                        ↓
                Random Forest Training
                        ↓
                Evaluation (accuracy, report, confusion matrix)
                        ↓
                ROC (optional)
                        ↓
                Real-time prediction


In [ ]:
import glob
import os
import numpy as np
import librosa.display
import soundfile as sf

# Dataset Selection

In [ ]:
import os
import platform

system = platform.system()

username = os.getlogin()

if system == "Linux":
    if "microsoft" in platform.uname().release.lower():
        data_directory = f"/mnt/c/Users/{username}/Desktop/RAVDESS"
    else:
        data_directory = f"/home/{username}/RAVDESS"

elif system == "Windows":
    data_directory = f"C:/Users/{username}/Desktop/RAVDESS"

print("Using dataset path:", data_directory)

if not os.path.exists(data_directory):
    raise FileNotFoundError(f"""
Dataset not found!

Expected path:
{data_directory}

Please update the path in Dataset Selection section.
""")

## 1.1 Dataset Emotion Distribution

In [ ]:
import pandas as pd

emotion_list = []

for file in glob.glob(data_directory + "/Actor_*/*.wav"):
    file_name = os.path.basename(file)
    emotion = emotions[file_name.split("-")[2]]
    emotion_list.append(emotion)

df = pd.DataFrame(emotion_list, columns=["emotion"])

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
sns.countplot(data=df, x="emotion")
plt.title("Emotion Distribution")
plt.show()

# 1.2 Waveform Visualization

In [ ]:
file = glob.glob(data_directory + "/Actor_*/*.wav")[0]

data, sr = librosa.load(file)

plt.figure(figsize=(12,4))
librosa.display.waveshow(data, sr=sr)
plt.title("Waveform")

# 1.3 Spectrogram Visualization

In [ ]:
X = librosa.stft(data)
Xdb = librosa.amplitude_to_db(abs(X))

plt.figure(figsize=(12,4))
librosa.display.specshow(Xdb, sr=sr, x_axis='time', y_axis='hz')
plt.colorbar()
plt.title("Spectrogram")

# 1.4 MFCC Visualization

In [ ]:
mfcc = librosa.feature.mfcc(y=data, sr=sr, n_mfcc=40)

plt.figure(figsize=(10,4))
librosa.display.specshow(mfcc, x_axis='time')
plt.colorbar()
plt.title("MFCC Features")

# Add More Features (Better than MFCC Only)

Your notebook uses:

```
MFCC
Chroma
Mel
```

Research models use additional features:

| Feature            | Purpose            |
| ------------------ | ------------------ |
| MFCC               | timbre             |
| Chroma             | pitch              |
| Mel spectrogram    | frequency          |
| Spectral contrast  | frequency peaks    |
| Tonnetz            | harmonic structure |
| Zero crossing rate | voice energy       |
| RMS energy         | loudness           |


In [ ]:
def extract_feature(data, sr):

    mfcc = np.mean(librosa.feature.mfcc(y=data, sr=sr, n_mfcc=40).T, axis=0)

    chroma = np.mean(librosa.feature.chroma_stft(y=data, sr=sr).T, axis=0)

    mel = np.mean(librosa.power_to_db(
            librosa.feature.melspectrogram(y=data, sr=sr)
        ).T, axis=0)

    contrast = np.mean(librosa.feature.spectral_contrast(y=data, sr=sr).T, axis=0)

    tonnetz = np.mean(librosa.feature.tonnetz(
        y=librosa.effects.harmonic(data), sr=sr).T, axis=0)

    zcr = np.mean(librosa.feature.zero_crossing_rate(data).T, axis=0)

    rms = np.mean(librosa.feature.rms(y=data).T, axis=0)

    return np.hstack([mfcc, chroma, mel, contrast, tonnetz, zcr, rms])

# 1.5 Feature Correlation Heatmap

In [ ]:
sample_features = [extract_feature(*librosa.load(f)) for f in glob.glob(data_directory + "/Actor_*/*.wav")[:50]]
df = pd.DataFrame(sample_features)

sns.heatmap(df.corr())

In [ ]:
emotion_files = {}

for file in glob.glob(data_directory + "/Actor_*/*.wav"):
    file_name = os.path.basename(file)
    emotion = emotions[file_name.split("-")[2]]

    if emotion not in emotion_files:
        emotion_files[emotion] = []
    
    emotion_files[emotion].append(file)

# Integrating Your Soft-Label Idea

Instead of:

```
neutral → [0,0,0,0,0,1,0,0]
```

Use:

```
neutral 0.7
angry 0.3

In [ ]:
emotion_map = {
'angry':0,
'calm':1,
'disgust':2,
'fearful':3,
'happy':4,
'neutral':5,
'sad':6,
'surprised':7
}

def create_soft_label(primary, secondary=None, w1=0.7, w2=0.3):

    label = np.zeros(8)

    if secondary:
        label[emotion_map[primary]] = w1
        label[emotion_map[secondary]] = w2
    else:
        label[emotion_map[primary]] = 1.0

    return label

# Automatically Generate Mixed Emotion Audio

```
Any sentence + Any word = mixed emotion audio
```

### Example:

```
neutral sentence + angry word = mixed emotion audio
```

In [ ]:
def mix_multiple_audio(files, weights):

    signals = []
    sr = None

    for file in files:
        data, sr = librosa.load(file)
        signals.append(data)

    # Match lengths
    min_len = min([len(s) for s in signals])
    signals = [s[:min_len] for s in signals]

    # Weighted sum
    mixed = np.zeros(min_len)

    for s, w in zip(signals, weights):
        mixed += w * s

    # Normalize
    mixed = mixed / (np.max(np.abs(mixed)) + 1e-6)

    return mixed, sr

# Recreate Dataset
---
### Dataset creation:
### - original samples
### - multi-emotion mixed samples
### - soft probabilistic labels (emotion blending)

In [ ]:
np.random.seed(42)

In [ ]:
X, y = [], []

emotion_list = list(emotion_map.keys())

for file in glob.glob(data_directory + "/Actor_*/*.wav"):

    # -------- ORIGINAL SAMPLE --------
    data, sr = librosa.load(file)
    feature = extract_feature(data, sr)

    file_name = os.path.basename(file)
    primary_emotion = emotions[file_name.split("-")[2]]

    X.append(feature)
    y.append(create_soft_label(primary_emotion))

    # -------- MIXED SAMPLE --------
    if np.random.rand() < 0.4:   # 40% augmentation

        # pick 2 or 3 emotions randomly
        n_mix = np.random.choice([2,3])

        selected_emotions = np.random.choice(emotion_list, n_mix, replace=False)

        selected_files = [
            np.random.choice(emotion_files[e])
            for e in selected_emotions
        ]

        # random weights (sum = 1)
        weights = np.random.dirichlet(np.ones(n_mix))

        mixed_audio, sr = mix_multiple_audio(selected_files, weights)

        feature = extract_feature(mixed_audio, sr)
        X.append(feature)

        # create soft label
        label = np.zeros(8)
        for emo, w in zip(selected_emotions, weights):
            label[emotion_map[emo]] = w

        y.append(label)

X = np.array(X)
y = np.array(y)

print("Dataset size:", X.shape)

# Normalize Features

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

# Saving the Features

In [ ]:
import pickle

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("emotion_map.pkl", "wb") as f:
    pickle.dump(emotion_map, f)

# Data Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Convert soft labels → hard labels
y_hard = np.argmax(y, axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_hard, test_size=0.2, stratify=y_hard, random_state=42
)

# Model Train 

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Predictions
y_pred = rf.predict(X_test)

# Model Save

In [ ]:
import pickle

with open("rf_model.pkl", "wb") as f:
    pickle.dump(rf, f)

# Evaluation Matrics

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Convert soft labels → hard labels
y_true = y_test

# Predictions
y_pred_classes = rf.predict(X_test)

# Accuracy
print("Accuracy:", accuracy_score(y_true, y_pred_classes))

# Full report
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred_classes, target_names=list(emotion_map.keys())))

# confusion matrix visual

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(10,7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

labels = list(emotion_map.keys())
plt.xticks(np.arange(len(labels))+0.5, labels, rotation=45)
plt.yticks(np.arange(len(labels))+0.5, labels)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
import numpy as np

cm = confusion_matrix(y_true, y_pred_classes)

TP = np.diag(cm)
FP = np.sum(cm, axis=0) - TP
FN = np.sum(cm, axis=1) - TP
TN = np.sum(cm) - (FP + FN + TP)

TPR = TP / (TP + FN)   # Recall
FPR = FP / (FP + TN)
TNR = TN / (TN + FP)   # Specificity

print("\nTPR (Recall):", TPR)
print("FPR:", FPR)
print("TNR (Specificity):", TNR)

In [ ]:
print("\nAverage Metrics:")

print("Mean TPR (Recall):", np.mean(TPR))
print("Mean FPR:", np.mean(FPR))
print("Mean TNR:", np.mean(TNR))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("Precision:", precision_score(y_true, y_pred_classes, average='weighted'))
print("Recall:", recall_score(y_true, y_pred_classes, average='weighted'))
print("F1 Score:", f1_score(y_true, y_pred_classes, average='weighted'))

### Save new audio:

In [ ]:
emotions_sample = np.random.choice(list(emotion_map.keys()), 3, replace=False)
files = [np.random.choice(emotion_files[e]) for e in emotions_sample]
weights = np.random.dirichlet(np.ones(len(files)))

mixed, sr = mix_multiple_audio(files, weights)

sf.write("mixed_audio.wav", mixed, sr)

# Real-Time Emotion Detection

Record audio:

In [ ]:
import sounddevice as sd

def record_audio(duration=3, sr=22050):

    print("Speak now")

    audio = sd.rec(int(duration*sr),
                   samplerate=sr,
                   channels=1)

    sd.wait()

    return audio.flatten()

# Predict emotion:

In [ ]:
import pickle

with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open("emotion_map.pkl", "rb") as f:
    emotion_map = pickle.load(f)

audio = record_audio()

feature = extract_feature(audio, 22050)

feature = scaler.transform(feature.reshape(1, -1))  



prediction = rf.predict(feature)

inv_map = {v:k for k,v in emotion_map.items()}
emotion = inv_map[prediction[0]]

print("Emotion:", emotion)